### **Encoding**

Types of Encoding : 

- **One Hot Encoding** : This focusses on the word presence -> Is the word present or not? 
- **Bag of Words** : This focusses on word count -> how many times does the word appear ?
- **TF-IDF (Term Frequency – Inverse Document Frequency)**: This focusses on count + importance weighting -> How important is the word in this document ? This reduces the weight of common words and increase sthe weight of more innovative words. 



#### **One Hot Encoding**

- One Hot Encoding : This focusses on the word presence -> Is the word present or not? 
- This is document level representation of data.
- This is used in Logistic Regression and Naive Bayes classification. 
- We convert our data into 0 & 1s. (Even when a word is repeated it will be marked as 1 and not 2 there). 


We get the documents, we convert everything into OHE vector by taking the complete vocabulary (unique words from the full data) and then we write 0 & 1 basis the entire vocabulary. 
So for every document we will get a vector containing 0&1 . 
The size of the vocabulary determines the size of the vector. 
Vocab size of 10000, creates the vector size of 10000

In the document level representation, we do a single vectoral represenation of the entire document which is called document level representation. 

But for DL oriented OHE, we usually do a word level (sequence) represenation, which creates a single vector row for each word such that each row represents only one word, which means if your document has 10 words, you will have 10 rows and each row being the length equal to total vocabulary set. Dimension (row x column = length of document x length of total vocabulary)



Pros : 
- Easy to implement
- Simple binary representation 
- No training required
- Direct mapping 


Cons:
- Sparse matrix mostly zeroes -> memory waste
    - Sparse matrix means huge vector size but very little information
- Vocabulary size if is 10,000 words, then each document representation will be 10,000 columns long
    - high dimensionality vocabulary increase -> vector size increases
- No semantic understanding : similar words also seem unrelated
- OOV (Out of vocabulary) problem could not be handled for new words. 
    - If any new word comes up which is not part of the vocabulary then that particular document containing the extra word out of vocabulary can't be One hot encoded. 


In [1]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

In [2]:
documents = ["people watch movie", 
             "people watch cricket", 
             "people like movie", 
             "people like cricket"]

In [3]:
tokens = [sentence.lower().split() for sentence in documents]
tokens

[['people', 'watch', 'movie'],
 ['people', 'watch', 'cricket'],
 ['people', 'like', 'movie'],
 ['people', 'like', 'cricket']]

In [4]:
all_words = [[word] for sentence in tokens for word in sentence]
all_words

[['people'],
 ['watch'],
 ['movie'],
 ['people'],
 ['watch'],
 ['cricket'],
 ['people'],
 ['like'],
 ['movie'],
 ['people'],
 ['like'],
 ['cricket']]

In [5]:
encoder = OneHotEncoder(sparse_output=False) # sparse_output=False means the encoder returns a dense NumPy array instead of a sparse matrix. This makes the output easier. 

encoder.fit(all_words)

OneHotEncoder(sparse_output=False)

In [6]:

print("Vocabulary:", encoder.categories_[0])

Vocabulary: ['cricket' 'like' 'movie' 'people' 'watch']


In [7]:
for sentence in tokens:
    encoded = encoder.transform([[word] for word in sentence])
    print("\nSentence:", sentence)
    print(encoded)


Sentence: ['people', 'watch', 'movie']
[[0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0.]]

Sentence: ['people', 'watch', 'cricket']
[[0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]]

Sentence: ['people', 'like', 'movie']
[[0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]]

Sentence: ['people', 'like', 'cricket']
[[0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]]


#### **Bag of Words**

Bag of words captures how frequently a word is used in a meaningful content . If a word repeats more in a sentence, BoW assumes it is more important 
- Bag of Words : This focusses on word count -> how many times does the word appear ?
- The length of the vector generated will again be equal to the vocabulary length. 
- The vocabulary doesn't include the stop words like a, an, the, etc. 
- Here the vector will not just be a collection of 0s & 1s as there will be repetition too (exactly why BoW was used because we are interested in frequency of words which can be >1). Simply put up, we create a vector that has the occurence or frequency of words for each document. 

Pros:
- Easy to implement
    - Very simple logic : Just count word occurences
    - No complex maths needed
- Captures words frequency
    - Shows how often a word appears
    More occurence -> higher importance
- Works well before embedding for classical NLP tasks useful for text classification, spam detection, sentiment analysis (basic)
- No training is required , direct conversion from text -> numbers. 


Cons : 
- Ignores word order : Sentence meaning can change but BoW won't notice. Eg: Man bites dog and dog bites man will both have same BoW representation. 
- No semantic understanding : Simialr words are treated completely different example: like vs love no relation is captured here. 
- High dimensionality : Vocabulary size = vector size i.e. large dataset will have huge vectors. 
- Sparse representation : mostly values will be 0 only which makes it memory inefficient and wastage of compute. 
- OOV problem : new words can't be handled i.e. if a word is not in vocabulary the BoW encoding won't be generated for an exceptional word and will be erroneus. 
- Overemphasizes frequent words : repeated words may dominate. a sentence could be "Movie movie movie" will get high importance but will be meaningless. 
- BoW is simple and powerful for counting words but it fails to understand the meaning, order and context 

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
documents = ["people watch movie and watch movie again", 
             "people watch cricket and watch cricket",
             "people like movie and like movie a lot",
             "people like cricket"]

In [10]:
vectorizer = CountVectorizer()

In [11]:
bow_matrix = vectorizer.fit_transform(documents)

In [12]:
print("Vocabulary:", vectorizer.get_feature_names_out())

Vocabulary: ['again' 'and' 'cricket' 'like' 'lot' 'movie' 'people' 'watch']


In [13]:
print("BOW Matrix:\n", bow_matrix.toarray())

BOW Matrix:
 [[1 1 0 0 0 2 1 2]
 [0 1 2 0 0 0 1 2]
 [0 1 0 2 1 2 1 0]
 [0 0 1 1 0 0 1 0]]


#### **TF-IDF**

- TF-IDF (Term Frequency – Inverse Document Frequency) was invented by Google and it was ruling in that era before the advent of Neural networks.  
- This focusses on count + importance weighting -> How important is the word in this document ? 
- This reduces the weight of common words and increases the weight of more innovative words.
- We take the documents data, we create a vocabulary and then we define :
    - TF = occurences of the word in the sentence / total words in the given sentence. 
        - example : My name is Hemant, Hemant is learning AI. TF(Hemant) = 2/8
    - IDF = log ( (Total number of sentence) / (number of sentences containing that word) )
        - Example : I love ML
                    I love DL
                    I like GenAI 
                    IDF(Love) = log(3/2)
    - Ultimately we calculate the TF * IDF

- **Why TF-IDF** : 
    - TF was needed because more occurence = more importance in that document. 
    - IDF was needed because common words are not very useful. 
- **Why TF X IDF** : 
    - Because TF --> Importance in document, 
    - IDF --> Importance in corpus

- **Why log was needed in IDF?** :
    - Without log , rare words get extremely high values and log smoothens the values and keeps a balance. 
    - Log reduces extreme differences while keeping the importance order intact
    - Log compresses large values so rare words don't overpower everything. 
    - Withiout Log what could it be: 

        - Total documents = 1000
            - Case 1 : Rare word (appearing in 1 doc) IDF = 1000/1 = 1000
            - Case 2 : Medium freq word (appearing in 10 docs) IDF = 1000/10 = 100
            - Case 3 : Common Word (appearing in 500 docs) IDF = 1000/500 = 2
        - Problem : Values have extremes 1000 vs 100 vs 2. Rare word dominates everything and the model becomes unstable without Log. 
        - Solution : with log the range will be Log (1000)  = 6.9 --> log(100) = 4.6 --> log (2) = 0.69

- PROS :
    - Captures importance of words and not just counts. Rare words --> higher weight, common words --> lower weight. 
    - Reduces impact of common words. IDF reduces their weight
    - Better than BoW for relevance hence it helps in search engines(Google Indexation of the URLs), information retrieval, ranking documents. 
    - No training and no model learning is required and it works as a direct calculation. 
    - Simple and Interpretable. 

- CONS :
    - Ignores word order, same problem like BoW Example: Dog bites man and Man bites dog will have same representation. 
    - No semantic understanding i.e. can't understand the meaning. car vs automobile are treated completely differently. 
    - Sparse vector and high dimensional where vector size = vocabulary size hence has a lot of zeroes which makes it memory inefficient. 
    - Dones't handle the context, same word can have completely different meaning but this will produce the same vector like bank in river bank versus bank in finance...same vector different meaning. 
    - OOV problem  Can't properly handle the words not in original vocabulary hence will be ignored. 



In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:

# documents = [
#     "I love machine learning",
#     "I love deep learning",
#     "machine learning is amazing"
# ]

documents= ["people watch movie and watch movie again",
"people watch cricket and watch cricket",
"people like movie and like movie a lot",
"people like cricket"]

In [16]:
tfidf_vectorizer = TfidfVectorizer()

In [17]:
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

In [18]:
print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())

Vocabulary: ['again' 'and' 'cricket' 'like' 'lot' 'movie' 'people' 'watch']


In [19]:

print("TF-IDF Matrix:\n", tfidf_matrix.toarray())

TF-IDF Matrix:
 [[0.38771136 0.24747114 0.         0.         0.         0.61135218
  0.20232387 0.61135218]
 [0.         0.2684707  0.66322944 0.         0.         0.
  0.21949239 0.66322944]
 [0.         0.24747114 0.         0.61135218 0.38771136 0.61135218
  0.20232387 0.        ]
 [0.         0.         0.64043405 0.64043405 0.         0.
  0.42389674 0.        ]]


In [20]:
for vector in tfidf_matrix.toarray():
    print("\nTF-IDF Vector:", vector)


TF-IDF Vector: [0.38771136 0.24747114 0.         0.         0.         0.61135218
 0.20232387 0.61135218]

TF-IDF Vector: [0.         0.2684707  0.66322944 0.         0.         0.
 0.21949239 0.66322944]

TF-IDF Vector: [0.         0.24747114 0.         0.61135218 0.38771136 0.61135218
 0.20232387 0.        ]

TF-IDF Vector: [0.         0.         0.64043405 0.64043405 0.         0.
 0.42389674 0.        ]


### **Embeddings**

Embeddings can be generated using 2 methods :
- Classical Method : Based on the concepts of neural networks
    - word2vec : developed by google
        - Also used in Transformers 
        - Can be used in two ways 
            - Pre-trained : (By Google)
            - Custom : own data, own model 
        
- Latest method (SOTA) : These are most widely used these days in LLMs. 
    - Transformer based models
        - OpenAI Embeddings
        - Gemini Embeddings

#### **Word2vec**

Word2vec developed by Google in 2013, depends on the concepts of neural networks. 
- 300 features/dimensions, every value of the feature will be represented in the 300-d space and hence each document will be a vector. 

Suppose you have 2 sentences :
- I like this movie
- I love this film 

Basis OHE/BoW/TF-IDF encoding methods, the words like and love are not same, further the words film and movie are not same 

Basis the concepts of embeddings like and love would be similar and hnece the movie and film will also be same. Acc to embeddings the two sentences have some sort of similarity and the model understands this similarity. 

**Vocabulary is always case-sensitive i.e. Hemant and hemant are two different words in Vocabulary**

word2vec is a neural network based mathematical model. We have to train the model basis the concepts of supervised learning for which we need data and the data should be in the form of X & Y i.e. input and the output variables. Now the data is unstructured data (text based). To convert the unstructured data into the form of X & Y we need to know the concept of window size. 

**Window size** : This is indicative of "How many neighbours do we have to consider surrounding the word". 
- For a window size of 1, we will consider 1 neighbour both on the left and right side of the word in focus. Now the words considered as neighbours becomes the context i.e. X variable and the word in focus becomes Y variable i.e. Target/Output variable.  
    - for the sentence "I love deep learning", the left side items below are the target or the output variables and the right side words are the input variables(context) : 
        - I : love
        - love : I, deep
        - deep : love, learning
        - learning : deep
- For a window size of 2, we will consider 2 neighbour both on the left and right side of the word in focus. Now the words considered as neighbours becomes the context i.e. X variable and the word in focus becomes Y variable i.e. Target/Output variable.  
    - for the sentence "I love deep learning", the left side items below are the target or the output variables and the right side words are the input variables(context) : 
        - I : love, deep
        - love : I, deep, learning
        - deep : I, love, learning
        - learning : love, deep



There are 2 methods here which differentiate on the level of data preparation:

Data => Follow Hemant for Gen AI

- CBOW : Continuous bag of Words 

    - We will have 2 columns namely context(I/P) and target(O/P)
        - For window size = 1, 
        - **I/P : O/P**
        - Hemant: Follow
        - Follow, for : Hemant 
        - Hemant, Gen : for 
        - for, AI : Gen
        - Gen: AI
    - On the above data of i/p and o/p you can train the neural network

- Skipgrams : The Input and output will exactly be opposite for Skipgrams data 
    - We will have 2 columns namely context(I/P) and target(O/P)
        - For window size = 1, 
        - **I/P : O/P**
        - Follow: Hemant
        - Hemant : Follow, for
        - for : Hemant, Gen
        - Gen : for, AI
        - AI : Gen
    - On the above data of i/p and o/p you can train the neural network


## CBOW (Continuous Bag of Words) Model in Word2Vec

### Example Sentence and Vocabulary
- **Sentence**: "Follow Hemant for Gen AI"
- **Vocabulary**: {AI, Follow, for, Gen, Hemant}

### One-Hot Encoding (OHE) Vectors
Each word is represented as a one-hot encoded vector based on the vocabulary:
- AI ⇒ [1, 0, 0, 0, 0]
- Follow ⇒ [0, 1, 0, 0, 0]
- for ⇒ [0, 0, 1, 0, 0]
- Gen ⇒ [0, 0, 0, 1, 0]
- Hemant ⇒ [0, 0, 0, 0, 1]

### CBOW Training Data (Window Size = 1)
For a window size of 1, we create input-output pairs based on the sentence context:
- **Input: Output**
- Hemant → Follow
- [Follow, for] → Hemant
- [Hemant, Gen] → for
- [for, AI] → Gen
- Gen → AI

### Neural Network Architecture
- **Input Layer**: Takes one-hot encoded vectors of context words.
- **Hidden Layer**: Learns the word embeddings. (Note: Google's original Word2Vec typically used 300 neurons/dimensions in the hidden layer for embeddings.)
- **Output Layer**: Predicts the target word using softmax.

### Key Insights
- The hidden layer's weights represent the learned embeddings for the predicted output word.
- During training (forward and backward propagation), weights are adjusted to minimize prediction error, resulting in meaningful word embeddings.
- After training, each word gets a dense vector representation (e.g., 300-dimensional) that captures semantic relationships.
- The dimension of the word2vec vector will be equal to hidden layer size. As per google's pretrained size, it was 300 , hence word2vec has the dimension of 300. The size of hidden layer size is actually a hyperparameter which can be demanded as per need, but basis the official paper on the parameters they used, it was 300. 

Benefits or Pros and Cons of Word2vec
Pros:
- Efficient & Fast training because it is a shallow model (no deep layers)
- Semantic understanding : similar meaning words -> close vectors Example king ~ queen, car ~ vehicle. Captures similarity and relationships. 
- vector arithmetic : king-man+woman ~ queen
- Pre-trained embeddings available: Google news vectors (300 dimensions), easy to use
- General purpose embeddings and can be used in NLP pipelines, ML Models, clustering/search

Cons :
- Context independant => same word = same vector always . bank(river) = bank(finance)
- Fixed embeddings => vector would be static will not change according to sentence context
- No handling of OOV. For example the word Chatgpt will be completely unknown to google's model. 
- Can't handle subwords like : running, runner, runs..all these are treated separately no morphological understanding
- Word2vec has no attention, no deep context and no dynamic embeddings. 